# XOR: обучение многослойной сети

XOR возвращает `1`, когда входы различаются, и `0`, когда они равны. Эту зависимость нельзя разделить одной прямой, поэтому одного нейрона недостаточно: сеть использует скрытый слой, чтобы построить нелинейное представление входов.

Здесь выход кодирует два класса в формате one-hot: класс `0` — входы различаются, класс `1` — входы равны. Softmax преобразует два выходных значения в вероятности этих классов.


In [1]:
import numpy as np

from open_neuron.activations import sigmoid, softmax
from open_neuron.layers import Layer
from open_neuron.models import NeuralNetwork
from open_neuron.optimizers import Adam

np.set_printoptions(precision=3, suppress=True)

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([[0, 1], [1, 0], [1, 0], [0, 1]])
class_names = np.array(["входы различаются", "входы равны"])

print("Входы:")
print(X)
print("Ожидаемые классы:", class_names[np.argmax(y, axis=1)])


Входы:
[[0 0]
 [0 1]
 [1 0]
 [1 1]]
Ожидаемые классы: ['входы равны' 'входы различаются' 'входы различаются' 'входы равны']


## Архитектура сети

Первый `Layer` содержит два скрытых нейрона с сигмоидой. Их выходы позволяют следующему слою построить нелинейную границу для XOR. Второй `Layer` содержит по одному выходу на класс и применяет softmax.

Categorical cross-entropy сравнивает вероятности softmax с one-hot ответами. Весь набор из четырёх строк передаётся одним пакетом; фиксированный seed делает начальные веса и результат повторяемыми.


In [2]:
np.random.seed(7)
model = NeuralNetwork(
    [
        Layer(2, activation=sigmoid),
        Layer(2, activation=softmax),
    ],
    inputs_dim=X.shape[1],
    loss="categorical_crossentropy",
    optimizer=Adam(learning_rate=0.1),
)

model.fit(X, y, epochs=500, batch_size=4, shuffle=False)
probabilities = model.predict(X)
predicted_classes = np.argmax(probabilities, axis=1)
expected_classes = np.argmax(y, axis=1)

print("Вероятности классов [различаются, равны]:")
print(probabilities)
print("Предсказанные классы:", class_names[predicted_classes])
print("Ожидаемые классы:    ", class_names[expected_classes])


Training started with Overall Loss 0.6971 on Train Data
Epoch 500/500, Loss: 0.0012, Validation Loss: 0.0012 on Train Data
Training completed with Overall Loss 0.0012
Вероятности классов [различаются, равны]:
[[0.001 0.999]
 [0.999 0.001]
 [0.999 0.001]
 [0.002 0.998]]
Предсказанные классы: ['входы равны' 'входы различаются' 'входы различаются' 'входы равны']
Ожидаемые классы:     ['входы равны' 'входы различаются' 'входы различаются' 'входы равны']


В каждой строке вероятности двух классов в сумме дают `1`. `argmax` выбирает класс с большей вероятностью. Для успешного обучения предсказанные классы должны совпасть с ожидаемыми для всех четырёх комбинаций входов.
